### **Carga local de Indicadores_vano_clima**

Este cuaderno parte del archivo local `data/Indicadores_vano_clima_nueva.csv`, normaliza `FECHA` al formato `%Y-%m-%d %H:%M:%S` y usa Open-Meteo para completar únicamente los valores climáticos faltantes.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

FECHA_FORMAT = "%Y-%m-%d %H:%M:%S"

_cwd = Path.cwd().resolve()
_project_candidate = None
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / "src" / "chec_impacto").exists():
        _project_candidate = _candidate
        break
if _project_candidate is None:
    _project_candidate = _cwd
_BOOTSTRAP_SRC_PATH = _project_candidate / "src"
if str(_BOOTSTRAP_SRC_PATH) not in sys.path:
    sys.path.insert(0, str(_BOOTSTRAP_SRC_PATH))

from chec_impacto.notebook_support import resolve_project_root

PROJECT_ROOT = resolve_project_root(clone_if_missing=False)
INPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v1.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v2.csv"

Xdata = pd.read_csv(INPUT_PATH)
Xdata.index.name = "original_index"



In [ ]:
Xdata

## **Normalizar FECHA y convertir a UTC**


In [ ]:
import os
import time
import numpy as np
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from chec_impacto.data.climate import parse_local_bogota_to_utc

REQUIRED_COLUMNS = ["FECHA", "X1", "Y1", "X2", "Y2"]
missing_required = [column for column in REQUIRED_COLUMNS if column not in Xdata.columns]
if missing_required:
    raise ValueError(f"Faltan columnas requeridas: {missing_required}")


print("Normalizando FECHA...")
Xdata["FECHA"] = pd.to_datetime(Xdata["FECHA"], errors="coerce").dt.strftime(FECHA_FORMAT)
dt_utc = parse_local_bogota_to_utc(Xdata["FECHA"].astype(str), fecha_format=FECHA_FORMAT)

DF_ISO = Xdata.copy()
DF_ISO["event_time_iso_utc"] = dt_utc

invalid_dates = DF_ISO["event_time_iso_utc"].isna().sum()
print(f"Filas totales: {len(DF_ISO):,}")
print(f"Fechas inválidas: {invalid_dates:,}")

if invalid_dates:
    raise ValueError("Hay fechas inválidas en FECHA; revisa esas filas antes de llamar la API.")

DF_ISO[["FECHA", "event_time_iso_utc", "X1", "Y1", "X2", "Y2"]].head()


## **Diagnóstico de columnas climáticas faltantes**


In [ ]:
DF_ISO[["FECHA", "event_time_iso_utc", "X1", "Y1", "X2", "Y2"]].head()


## **Filas que necesitan completar clima**


In [ ]:
LAGS = 25  # t, t-1, ..., t-24

VAR_MAP = {
    "prep": "precipitation",
    "pres": "pressure_msl",
    "sp": "surface_pressure",
    "rh": "relative_humidity_2m",
    "solar_rad": "shortwave_radiation",
    "temp": "temperature_2m",
    "wind_gust_spd": "wind_gusts_10m",
    "wind_spd": "wind_speed_10m",
    "clouds": "cloud_cover",
}
VARS_SHORT = list(VAR_MAP.keys())
CLIMATE_COLUMNS = [f"{short}_{lag}" for short in VARS_SHORT for lag in range(LAGS)]

missing_climate_columns = [column for column in CLIMATE_COLUMNS if column not in DF_ISO.columns]
if missing_climate_columns:
    print(f"Se crearán {len(missing_climate_columns)} columnas climáticas que no existen en el archivo.")
    for column in missing_climate_columns:
        DF_ISO[column] = np.nan

rows_with_missing_climate = DF_ISO[CLIMATE_COLUMNS].isna().any(axis=1)
pending_index = DF_ISO.index[rows_with_missing_climate].tolist()

print(f"Columnas climáticas esperadas: {len(CLIMATE_COLUMNS):,}")
print(f"Filas con algún clima faltante: {len(pending_index):,}")
DF_ISO.loc[pending_index, ["FECHA", "X1", "Y1", "X2", "Y2"] + CLIMATE_COLUMNS[:5]].head()


## **Completar datos climáticos faltantes con Open-Meteo**


In [ ]:
ALL_API_VARS = [VAR_MAP[k] for k in VAR_MAP]
ARCHIVE_SAFE_API_VARS = [v for v in ALL_API_VARS]


### **Helpers**

In [ ]:
# ==== Helpers ====
from chec_impacto.data.climate import (
    combine_payloads as _combine_payloads,
    get_hourly_window as fetch_hourly_window,
    hour_floor,
    slice_time_window as _slice_time_window,
    to_utc_iso,
)


## **Configuración de API y archivo de salida**


In [ ]:
import requests_cache
from retry_requests import retry
import openmeteo_requests
from tqdm.auto import tqdm

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
ARCHIVE_URL  = "https://archive-api.open-meteo.com/v1/archive"

SAVE_EVERY_ROWS = 100
REQUEST_SLEEP_SECONDS = 0.35
MAX_ROWS_TO_PROCESS = None  # Usa un entero para probar con pocas filas, por ejemplo 10.

cache_session = requests_cache.CachedSession(str(PROJECT_ROOT / "data" / ".openmeteo_cache"), expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

STOP_FILE = PROJECT_ROOT / "data" / "STOP_OPENMETEO_LIMIT.txt"
if STOP_FILE.exists():
    raise RuntimeError(
        f"Se encontró {STOP_FILE}. Ya se detectó un límite en una ejecución previa. "
        "Borra ese archivo si deseas reintentar."
    )

if OUTPUT_PATH.exists():
    print(f"Se encontró un archivo de salida previo. Se continuará desde: {OUTPUT_PATH}")
    DF_FILLED = pd.read_csv(OUTPUT_PATH)
    DF_FILLED["FECHA"] = pd.to_datetime(DF_FILLED["FECHA"], errors="coerce").dt.strftime(FECHA_FORMAT)
    DF_FILLED["event_time_iso_utc"] = parse_local_bogota_to_utc(DF_FILLED["FECHA"].astype(str), fecha_format=FECHA_FORMAT)
    for column in CLIMATE_COLUMNS:
        if column not in DF_FILLED.columns:
            DF_FILLED[column] = np.nan
else:
    DF_FILLED = DF_ISO.copy()

pending_mask = DF_FILLED[CLIMATE_COLUMNS].isna().any(axis=1)
pending_index = DF_FILLED.index[pending_mask].tolist()
if MAX_ROWS_TO_PROCESS is not None:
    pending_index = pending_index[:MAX_ROWS_TO_PROCESS]

print(f"Archivo de entrada: {INPUT_PATH}")
print(f"Archivo de salida: {OUTPUT_PATH}")
print(f"Filas pendientes a consultar en API: {len(pending_index):,}")


In [ ]:
DF_FILLED.loc[pending_index, ["FECHA", "X1", "Y1", "X2", "Y2"] + CLIMATE_COLUMNS[:5]].head()


In [ ]:
from chec_impacto.data.climate import save_completed_dataset

processed_rows = 0
stop_all = False

for idx in tqdm(pending_index, desc="Completando clima", unit="fila"):
    lon = pd.to_numeric(DF_FILLED.at[idx, "X1"], errors="coerce")
    lat = pd.to_numeric(DF_FILLED.at[idx, "Y1"], errors="coerce")
    event_utc = pd.to_datetime(DF_FILLED.at[idx, "event_time_iso_utc"], utc=True, errors="coerce")

    if pd.isna(event_utc) or pd.isna(lat) or pd.isna(lon):
        continue

    try:
        event_hour_utc = hour_floor(event_utc.to_pydatetime())
        start_dt = event_hour_utc - timedelta(hours=24)
        end_dt = event_hour_utc
        payload = fetch_hourly_window(
            openmeteo,
            float(lat),
            float(lon),
            start_dt,
            end_dt,
            archive_url=ARCHIVE_URL,
            forecast_url=FORECAST_URL,
            archive_vars=ARCHIVE_SAFE_API_VARS,
            forecast_vars=ALL_API_VARS,
        )
    except Exception as exc:
        msg = str(exc)
        is_limit = (
            "Daily API request limit exceeded" in msg
            or "Hourly API request limit exceeded" in msg
            or "rate limit" in msg.lower()
            or "too many requests" in msg.lower()
            or "429" in msg
        )
        if is_limit:
            STOP_FILE.write_text(f"Open-Meteo limit detected | index={idx}\n{msg}\n")
            print(f"Límite de API detectado en índice {idx}. Se guarda avance parcial y se detiene.")
            stop_all = True
            break

        print(f"Error en índice {idx}: {exc}")
        continue

    desired_times = [event_hour_utc - timedelta(hours=lag) for lag in range(LAGS)]
    desired_iso = [to_utc_iso(t) for t in desired_times]
    time_index = [to_utc_iso(pd.Timestamp(t).to_pydatetime()) for t in payload["time"]]
    pos = {ts: position for position, ts in enumerate(time_index)}

    any_filled = False
    for short_name, api_name in VAR_MAP.items():
        series_vals = payload.get(api_name)
        if series_vals is None:
            continue

        for lag, ts in enumerate(desired_iso):
            column = f"{short_name}_{lag}"
            if not pd.isna(DF_FILLED.at[idx, column]):
                continue

            position = pos.get(ts)
            if position is not None:
                DF_FILLED.at[idx, column] = series_vals[position]
                any_filled = True

    if any_filled:
        processed_rows += 1

    if processed_rows and processed_rows % SAVE_EVERY_ROWS == 0:
        save_completed_dataset(DF_FILLED, OUTPUT_PATH)
        print(f"Avance guardado: {processed_rows:,} filas completadas en esta ejecución.")

    time.sleep(REQUEST_SLEEP_SECONDS)

save_completed_dataset(DF_FILLED, OUTPUT_PATH)

remaining_rows = DF_FILLED[CLIMATE_COLUMNS].isna().any(axis=1).sum()
print(f"Archivo guardado en: {OUTPUT_PATH}")
print(f"Filas completadas en esta ejecución: {processed_rows:,}")
print(f"Filas con clima pendiente después de guardar: {remaining_rows:,}")

if stop_all:
    print("Ejecución detenida por límite de API. Puedes reintentar después de borrar STOP_OPENMETEO_LIMIT.txt.")


In [ ]:
# Reporte de vacíos después de completar con API, antes de la limpieza final.
API_REPORT_SOURCE = DF_FILLED.drop(columns=["event_time_iso_utc"], errors="ignore")
api_rows = len(API_REPORT_SOURCE)
api_missing = API_REPORT_SOURCE.isna().sum()
reporte_vacios_api = (
    pd.DataFrame({
        "columna": api_missing.index,
        "vacios_api": api_missing.values,
        "porcentaje_vacios_api": api_missing.values / api_rows * 100,
    })
    .query("vacios_api > 0")
    .sort_values(["porcentaje_vacios_api", "vacios_api", "columna"], ascending=[False, False, True])
    .reset_index(drop=True)
)

print("Reporte después de API, antes de limpieza final")
print(f"Filas evaluadas: {api_rows:,}")
print(f"Columnas con vacíos > 0: {len(reporte_vacios_api):,}")
display(reporte_vacios_api)

# Limpieza final y generación del dataset v2.
# OUTPUT_PATH conserva el resultado completado por API.
# FINAL_OUTPUT_PATH guarda el resultado final después de imputar vacíos y eliminar ALTURA vacía.
FINAL_OUTPUT_PATH = PROJECT_ROOT / "data" / "Indicadores_vano_v3.csv"
NUMERIC_ZERO_FILL_COLUMNS = [
    "PROMEDIO_KWH_VANO",
    "PROMEDIO_KWH_TRF",
    "NR_T",
    "CNT_USUS",
    "CAPACIDAD_NOMINAL",
]
TEXT_ZERO_FILL_COLUMNS = ["CALIBRE_NEUTRO"]
ZERO_FILL_COLUMNS = NUMERIC_ZERO_FILL_COLUMNS + TEXT_ZERO_FILL_COLUMNS
FECHA_TRF_COLUMN = "FECHA_OPERACION_TRF"
FECHA_VANO_COLUMN = "FECHA_OPERACION_VANO"
ALTURA_COLUMN = "ALTURA"

required_final_columns = ZERO_FILL_COLUMNS + [FECHA_TRF_COLUMN, FECHA_VANO_COLUMN, ALTURA_COLUMN]
missing_final_columns = [column for column in required_final_columns if column not in DF_FILLED.columns]
if missing_final_columns:
    raise ValueError(f"Faltan columnas requeridas para la limpieza final: {missing_final_columns}")

DF_CLEAN = DF_FILLED.copy()

zero_fill_summary = {}
for column in NUMERIC_ZERO_FILL_COLUMNS:
    original_series = DF_CLEAN[column]
    blank_mask = original_series.isna() | original_series.astype(str).str.strip().eq("")
    zero_fill_summary[column] = int(blank_mask.sum())
    DF_CLEAN[column] = pd.to_numeric(original_series, errors="coerce").fillna(0)

for column in TEXT_ZERO_FILL_COLUMNS:
    blank_mask = DF_CLEAN[column].isna() | DF_CLEAN[column].astype(str).str.strip().eq("")
    zero_fill_summary[column] = int(blank_mask.sum())
    DF_CLEAN[column] = DF_CLEAN[column].mask(blank_mask, "0")

fecha_trf_blank_mask = DF_CLEAN[FECHA_TRF_COLUMN].isna() | DF_CLEAN[FECHA_TRF_COLUMN].astype(str).str.strip().eq("")
fecha_trf_replacements = int(fecha_trf_blank_mask.sum())
DF_CLEAN.loc[fecha_trf_blank_mask, FECHA_TRF_COLUMN] = DF_CLEAN.loc[fecha_trf_blank_mask, FECHA_VANO_COLUMN]

rows_before_altura = len(DF_CLEAN)
DF_CLEAN = DF_CLEAN.dropna(subset=[ALTURA_COLUMN]).copy()
rows_removed_altura = rows_before_altura - len(DF_CLEAN)

# Guardar el resultado final v3. No se sobreescribe OUTPUT_PATH.
save_completed_dataset(DF_CLEAN, FINAL_OUTPUT_PATH)

# Reporte de vacíos después de limpieza final: solo se muestra en el notebook.
FINAL_REPORT_SOURCE = DF_CLEAN.drop(columns=["event_time_iso_utc"], errors="ignore")
final_rows = len(FINAL_REPORT_SOURCE)
final_missing = FINAL_REPORT_SOURCE.isna().sum()
reporte_vacios_final = (
    pd.DataFrame({
        "columna": final_missing.index,
        "vacios_final": final_missing.values,
        "porcentaje_vacios_final": final_missing.values / final_rows * 100,
    })
    .query("vacios_final > 0")
    .sort_values(["porcentaje_vacios_final", "vacios_final", "columna"], ascending=[False, False, True])
    .reset_index(drop=True)
)




In [ ]:
reporte_vacios_final

In [ ]:
DF_CLEAN